# Waimea WaveCast — Open-Meteo Marine: toàn bộ biến, lịch sử từ 01/09/2026 đến hiện tại

**Mục đích:** tải toàn bộ trường dữ liệu Marine mà Open-Meteo hỗ trợ cho Waimea Bay (`21.671, -158.118`), **không** sử dụng `past_hours`, `forecast_hours` hay cắt cửa sổ thời gian xuống vài giờ. Lưu dữ liệu thô JSON (Bronze) và CSV đầy đủ (Silver ban đầu).

- `hourly`: 22 biến Marine chính thức, đầy đủ mọi mốc theo giờ mà API trả về.
- `daily`: 11 trường tổng hợp theo ngày, lưu riêng vì có độ phân giải khác `hourly`.
- `invert_barometer_height`: thử độc lập vì tài liệu giao diện liệt kê thêm biến này nhưng phiên bản OpenAPI có thể khác.
- `minutely_15`: thử 3 biến dòng chảy/mực nước ở độ phân giải 15 phút. **Không phải dữ liệu sóng đo 15 phút; một số khu vực là nội suy từ hourly.** Nếu không hỗ trợ toàn bộ khoảng lịch sử, notebook báo lỗi và vẫn giữ hourly/daily.

**Lưu ý:** Đây là **mô hình hải dương**, không phải quan trắc phao. Thời điểm cuối là **thời gian chạy notebook trên máy bạn (UTC)**. Khoảng lịch sử gần nhất của Marine Forecast API có giới hạn (tài liệu nêu `past_days` tối đa 92); nếu lùi xa hơn, cần đổi sang nguồn lịch sử tương ứng.

Tham khảo: https://open-meteo.com/en/docs/marine-weather-api và https://github.com/open-meteo/open-meteo/blob/main/openapi/marine.yml

## 1. Cài thư viện (chạy một lần)

In [ ]:
%pip install -q requests pandas matplotlib

## 2. Cấu hình ngày, tọa độ và toàn bộ biến Marine

`END_DATE` tự lấy ngày UTC hiện tại; sau khi tải sẽ **lọc bỏ mọi thời điểm vượt quá thời điểm đang chạy**. Không quy định độ dài cố định 7 giờ/24 giờ.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import time

import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

URL = "https://marine-api.open-meteo.com/v1/marine"
LATITUDE, LONGITUDE = 21.671, -158.118
START_DATE = "2026-09-01"
NOW_UTC = pd.Timestamp.now(tz="UTC")
END_DATE = NOW_UTC.date().isoformat()
OUT = Path("waimea_open_meteo_history")
OUT.mkdir(parents=True, exist_ok=True)

# Toàn bộ 22 biến hourly trong tài liệu Open-Meteo Marine hiện hành
HOURLY_VARS = [
    "wave_height", "wave_direction", "wave_period", "wave_peak_period",
    "wind_wave_height", "wind_wave_direction", "wind_wave_period", "wind_wave_peak_period",
    "swell_wave_height", "swell_wave_direction", "swell_wave_period", "swell_wave_peak_period",
    "secondary_swell_wave_height", "secondary_swell_wave_period", "secondary_swell_wave_direction",
    "tertiary_swell_wave_height", "tertiary_swell_wave_period", "tertiary_swell_wave_direction",
    "sea_level_height_msl", "sea_surface_temperature",
    "ocean_current_velocity", "ocean_current_direction",
]

# Toàn bộ các trường tổng hợp hằng ngày đã công bố
DAILY_VARS = [
    "wave_height_max", "wave_direction_dominant", "wave_period_max",
    "wind_wave_height_max", "wind_wave_direction_dominant", "wind_wave_period_max",
    "wind_wave_peak_period_max", "swell_wave_height_max",
    "swell_wave_direction_dominant", "swell_wave_period_max", "swell_wave_peak_period_max",
]

MINUTELY_15_VARS = ["ocean_current_velocity", "ocean_current_direction", "sea_level_height_msl"]

BASE_PARAMS = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "timezone": "UTC",
    "cell_selection": "sea",
}

print("Khoảng yêu cầu:", START_DATE, "→", END_DATE)
print("Thời gian chạy (UTC):", NOW_UTC)
print("Tổng biến hourly:", len(HOURLY_VARS), "| daily:", len(DAILY_VARS))
if (NOW_UTC - pd.Timestamp(START_DATE, tz="UTC")).days > 91:
    print("CẢNH BÁO: Khoảng này có thể vượt giới hạn lịch sử gần đây (~92 ngày) của Marine Forecast API.")

## 3. HTTP client có timeout và retry

Nếu API từ chối một biến hoặc khoảng lịch sử, in nguyên thông báo lỗi để có thể sửa chính xác; **không âm thầm bỏ cột**.

In [ ]:
session = requests.Session()
retry = Retry(
    total=3,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retry))


def fetch_marine(extra_params: dict, label: str) -> dict:
    params = {**BASE_PARAMS, **extra_params}
    response = session.get(URL, params=params, timeout=(10, 120))
    print(f"[{label}] HTTP {response.status_code} | URL: {response.url}")
    try:
        payload = response.json()
    except ValueError:
        raise RuntimeError(f"{label}: API trả về nội dung không phải JSON: {response.text[:400]}")
    if not response.ok or payload.get("error"):
        raise RuntimeError(f"{label}: {payload.get('reason', response.text[:700])}")
    (OUT / f"{label}_raw.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return payload


def time_table(payload: dict, key: str, filter_future: bool = True) -> pd.DataFrame:
    block = payload.get(key)
    if not block or "time" not in block:
        raise ValueError(f"Không có phần '{key}' trong kết quả API")
    df = pd.DataFrame(block)
    df = df.rename(columns={"time": "timestamp_utc"})
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
    if filter_future:
        df = df.loc[df["timestamp_utc"] <= NOW_UTC].copy()
    df = df.sort_values("timestamp_utc").drop_duplicates("timestamp_utc")
    df.insert(1, "timestamp_vietnam", df["timestamp_utc"].dt.tz_convert("Asia/Ho_Chi_Minh"))
    df.insert(2, "timestamp_hawaii", df["timestamp_utc"].dt.tz_convert("Pacific/Honolulu"))
    return df.reset_index(drop=True)

## 4. Tải **tất cả biến hourly** + **tất cả biến daily** trong một request

In [ ]:
payload = fetch_marine(
    {"hourly": ",".join(HOURLY_VARS), "daily": ",".join(DAILY_VARS)},
    label="marine_hourly_daily",
)

hourly_df = time_table(payload, "hourly")
daily_df = time_table(payload, "daily")

hourly_df.to_csv(OUT / "open_meteo_all_hourly.csv", index=False, encoding="utf-8-sig")
daily_df.to_csv(OUT / "open_meteo_all_daily.csv", index=False, encoding="utf-8-sig")
(OUT / "hourly_units.json").write_text(json.dumps(payload.get("hourly_units", {}), indent=2), encoding="utf-8")
(OUT / "daily_units.json").write_text(json.dumps(payload.get("daily_units", {}), indent=2), encoding="utf-8")

print("Hourly:", hourly_df.shape, "|", hourly_df["timestamp_utc"].min(), "→", hourly_df["timestamp_utc"].max())
print("Daily :", daily_df.shape, "|", daily_df["timestamp_utc"].min(), "→", daily_df["timestamp_utc"].max())
print("Số biến hourly thực nhận:", len([c for c in hourly_df.columns if c not in {"timestamp_utc", "timestamp_vietnam", "timestamp_hawaii"}]))
print("Dữ liệu trả về tại tọa độ lưới:", payload.get("latitude"), payload.get("longitude"))
display(hourly_df)

## 5. Thử thêm biến Inverted Barometer Height (nếu backend hỗ trợ)

Được liệt kê trong phần tùy chọn của trang Marine API, nhưng không có trong một số bản OpenAPI. Không làm mất dữ liệu chính nếu gặp HTTP 400.

In [ ]:
extra_variable = "invert_barometer_height"
try:
    extra_payload = fetch_marine({"hourly": extra_variable}, "inverted_barometer")
    extra_df = time_table(extra_payload, "hourly")
    extra_df = extra_df[["timestamp_utc", extra_variable]]
    hourly_df = hourly_df.merge(extra_df, on="timestamp_utc", how="left", validate="one_to_one")
    hourly_df.to_csv(OUT / "open_meteo_all_hourly.csv", index=False, encoding="utf-8-sig")
    print("Đã thêm cột:", extra_variable)
except (requests.RequestException, RuntimeError, ValueError) as exc:
    print("Không thêm được biến tùy chọn:", exc)

## 6. Thử dữ liệu 15 phút (nếu mô hình, khu vực và khoảng thời gian hỗ trợ)

Ba biến: dòng chảy (vận tốc, hướng) và mực nước. **Không nội suy độ cao sóng 1 giờ thành 15 phút.**

In [ ]:
minutely_15_df = None
try:
    p15 = fetch_marine({"minutely_15": ",".join(MINUTELY_15_VARS)}, "marine_15min")
    minutely_15_df = time_table(p15, "minutely_15")
    minutely_15_df.to_csv(OUT / "open_meteo_15min.csv", index=False, encoding="utf-8-sig")
    print("15-minute:", minutely_15_df.shape)
    print("Khoảng thời gian:", minutely_15_df["timestamp_utc"].min(), "→", minutely_15_df["timestamp_utc"].max())
    print("LƯU Ý: Một số khu vực, dữ liệu 15 phút là nội suy từ mô hình hourly.")
except (requests.RequestException, RuntimeError, ValueError) as exc:
    print("Không lấy được dữ liệu 15 phút cho toàn bộ khoảng yêu cầu:", exc)
    print("Dữ liệu hourly/daily đã lưu an toàn; không thay đổi độ phân giải để lấp chỗ thiếu.")

## 7. Kiểm chứng: khoảng thời gian, độ phân giải thực nhận, số giá trị thiếu

In [ ]:
print("===== KIỂM TRA HOURLY =====")
print("Số dòng:", len(hourly_df))
print("Số cột:", len(hourly_df.columns))
print("Từ (UTC):", hourly_df["timestamp_utc"].min())
print("Đến (UTC):", hourly_df["timestamp_utc"].max())
print("Thời điểm tải (UTC):", NOW_UTC)
print("Các khoảng cách giữa mốc thời gian nhận được:")
print(hourly_df["timestamp_utc"].diff().value_counts(dropna=True).to_string())

missing = pd.DataFrame({
    "column": [c for c in hourly_df.columns if not c.startswith("timestamp_")],
    "missing_values": [int(hourly_df[c].isna().sum()) for c in hourly_df.columns if not c.startswith("timestamp_")],
})
missing["missing_percent"] = (missing["missing_values"] / max(len(hourly_df), 1) * 100).round(2)
display(missing.sort_values("missing_percent", ascending=False).reset_index(drop=True))

print("===== 5 BẢN GHI CUỐI =====")
display(hourly_df.tail(5))
print("Dữ liệu hourly đầy đủ đã lưu vào:", (OUT / "open_meteo_all_hourly.csv").resolve())

## 8. Biểu đồ trực quan độ cao sóng từ tháng 9 đến thời điểm tải

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hourly_df["timestamp_utc"], hourly_df["wave_height"])
ax.set(title="Open-Meteo Marine — Wave Height | Waimea Bay", xlabel="Time (UTC)", ylabel="Wave height (m)")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Tệp đầu ra

- `waimea_open_meteo_history/marine_hourly_daily_raw.json`: phản hồi nguyên gốc cho Bronze.
- `waimea_open_meteo_history/open_meteo_all_hourly.csv`: **toàn bộ bản ghi hourly** từ 01/09 đến thời điểm chạy, mọi cột hỗ trợ.
- `waimea_open_meteo_history/open_meteo_all_daily.csv`: kết quả daily đầy đủ (lưu riêng).
- `waimea_open_meteo_history/open_meteo_15min.csv`: chỉ có nếu API cung cấp được dữ liệu 15 phút.
- `waimea_open_meteo_history/hourly_units.json` và `daily_units.json`: đơn vị biến.

**Không tự tạo bản ghi đo vào những thời điểm không có dữ liệu gốc.** Nếu có `NaN`, đó là biến thiếu từ mô hình/nguồn tại địa điểm hoặc thời gian đó. Bạn có thể chạy lại notebook vào ngày khác để kéo dài tới thời điểm mới nhất; không cần sửa ngày kết thúc.